In [30]:
%pwd


'C:\\Users\\tanni\\OneDrive\\Desktop\\CH'

In [31]:
import os 
os.chdir("../")

In [32]:
import os

os.chdir(r"C:\Users\tanni\OneDrive\Desktop\CH")

print(os.getcwd())

C:\Users\tanni\OneDrive\Desktop\CH


In [33]:
import langchain
import langchain_core
import langchain_community

print("LangChain:", langchain.__version__)
print("Core:", langchain_core.__version__)
print("Community:", langchain_community.__version__)

LangChain: 0.3.26
Core: 0.3.86
Community: 0.3.26


In [34]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

import os
def load_pdf(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

extracted_data = load_pdf("data")
print(len(extracted_data))
print(extracted_data[0].page_content[:500])

334
Wargo, Donald T.
Book
Economics for Life: Real-World Financial Literacy
Provided in Cooperation with:
Temple University Press
Suggested Citation: Wargo, Donald T. (2023) : Economics for Life: Real-World Financial Literacy, ISBN
9781439919842, Temple University Press, Philadelphia, PA,
https://doi.org/10.34944/2gyr-1m17
This Version is available at:
https://hdl.handle.net/10419/281347
Standard-Nutzungsbedingungen:
Die Dokumente auf EconStor dürfen zu eigenen wissenschaftlichen
Zwecken und zum Pri


In [35]:
from typing import List
from langchain_core.documents import Document

def filter_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")  # metadata key is usually lowercase

        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs

In [36]:
#split the documents into smaller chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter
def text_splitter(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=26,
        length_function=len
    ) 
    texts= text_splitter.split_documents(minimal_docs)
    return texts

In [37]:
minimal_docs = filter_minimal_docs(extracted_data)
texts = text_splitter(minimal_docs)
print(f"Number of chunks: {len(texts)}")
print(f"Number of chunks: {len(texts)}")


Number of chunks: 1232
Number of chunks: 1232


In [38]:
# import sys

# !{sys.executable} -m pip uninstall -y protobuf
# !{sys.executable} -m pip install protobuf==5.29.5

In [39]:
# embeddings
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    embeddings=HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings
embeddings=download_embeddings()

In [40]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Get API keys
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Set environment variables (optional but commonly used)
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [41]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [42]:
from pinecone import ServerlessSpec
index_name="finance-coach"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )
index = pc.Index(index_name)

In [43]:
from langchain_pinecone import PineconeVectorStore
docsearch=PineconeVectorStore.from_documents(
    documents=texts,
    embedding=embeddings,
    index_name=index_name
)
#load Existing Index
docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [44]:
retriever=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})
retrieved_docs = retriever.invoke("i want to buy car?")
retrieved_docs

[Document(id='d730e886-bed9-487b-8732-38d79fad62e8', metadata={'source': 'data\\Temple-University-Press_9781439919842.pdf'}, page_content='Union. \n2. Finance your loan over 60 months in order to bring down your monthly \npayments. \n3. Do not load your car up with customizations. \n4. Buy a used car with a warranty instead of a new car. \nYou should frst establish a monthly budget, keeping in mind the user costs. \nThen make a list of the few cars that will ft that budget. Drive the three cars that \nft your budget and choose the one your gut tells you that you like the most. \nThat way you will be happy with the purchase.'),
 Document(id='b0f30431-b8f8-42c7-a4b7-8f421fccfac7', metadata={'source': 'data\\Temple-University-Press_9781439919842.pdf'}, page_content='Union. \n2. Finance your loan over 60 months in order to bring down your monthly \npayments. \n3. Do not load your car up with customizations. \n4. Buy a used car with a warranty instead of a new car. \nYou should frst establi

In [45]:
# !pip install langchain-google-genai==2.1.8
# !pip install protobuf==6.33.0

In [46]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [47]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a Finance coach for answering data. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

In [48]:
from langchain.chains.combine_documents import create_stuff_documents_chain
prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}")
    ]
)
question_answer_chain=create_stuff_documents_chain(chatModel,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer_chain)

In [51]:
response = rag_chain.invoke({"input": "acne ?"})
print(response["answer"])

Under the ACA, insurance companies cannot refuse healthcare coverage due to a pre-existing condition. This means that if acne is considered a pre-existing condition, it cannot be a reason to deny you coverage.
